# Thermal Stack

Le pipeline est dans `thermal_stack.py`. Les donnees marche (spot, consommation 30 min, fuel cocktail) sont chargees via `data_loader.py`.

In [ ]:
import pandas as pd
from IPython.display import display

from thermal_stack import (
    ThermalStackConfig,
    build_thermal_stack,
    merit_order_for_areas,
    plot_merit_order_plotly,
    plot_regional_merit_order_plotly,
    identify_price_setters,
    summarize_price_setters,
    plot_price_setter_share_plotly,
    PriceSetterConfig,
    default_price_setter_configs,
    run_price_setter_configs,
    summarize_price_setter_configs,
    compare_price_setter_demand_levels,
    plot_price_setter_config_comparison,
    KANSAI_FENCE_AREAS,
)
import data_loader

In [ ]:
result = build_thermal_stack(ThermalStackConfig(delivery_month="2026-04", export_excel=False))
print(f"Delivery month: {result.delivery_month:%Y-%m}")
print(f"Thermal stack rows: {len(result.merit_order):,}")

## Merit Order

In [ ]:
region = "Tokyo"
mo_region = merit_order_for_areas(result, areas=region)
median_snapshot = identify_price_setters(mo_region, region=region).loc[lambda df: df.index.month == result.delivery_month.month]
median_demand = median_snapshot["demand_gw"].median()
median_spot = median_snapshot["spot_eur_mwh"].median()

fig = plot_merit_order_plotly(
    mo_region,
    region=region,
    period=result.delivery_month,
    demand_gw=median_demand,
    spot_price_eur_mwh=median_spot,
    title=f"Merit Order ({region}) - {result.delivery_month:%B %Y}",
)
fig.show()

In [ ]:
fig_regional = plot_regional_merit_order_plotly(result)
fig_regional.show()

## Marginal price setter - configurations paralleles

Comparaison de 6 configurations:
- **Region**: Kansai seul vs fence **Kansai + Hokuriku + Chubu**
- **Demande**:
  - `consumption` = consommation brute
  - `residual_load` = consommation - solaire - eolien
  - `thermal_residual` = consommation - solaire - eolien - inter_flows - hydro - nucleaire

Le prix spot de reference reste **Kansai** (`jpKan`) pour toutes les configs fence.

In [ ]:
configs = default_price_setter_configs(include_kansai_fence=True)
config_results = run_price_setter_configs(
    result,
    configs=configs,
    start=result.delivery_month,
    end=result.delivery_month + pd.offsets.MonthEnd(0),
)

demand_comparison = compare_price_setter_demand_levels(config_results)
summary_comparison = summarize_price_setter_configs(config_results)

display(demand_comparison.round(2))
display(summary_comparison.round(3))

In [ ]:
fig_compare = plot_price_setter_config_comparison(
    summary_comparison,
    title=f"Marginal fuel share by configuration - {result.delivery_month:%B %Y}",
)
fig_compare.show()

In [ ]:
# Focus Kansai: impact du fence et de la base de demande
kansai_configs = [cfg for cfg in configs if "Kansai" in cfg.name]
for cfg in kansai_configs:
    summary = summarize_price_setters(config_results[cfg.name])
    fig = plot_price_setter_share_plotly(
        summary,
        region=cfg.name,
        title=f"{cfg.name} - {result.delivery_month:%B %Y}",
    )
    fig.show()

In [ ]:
# Merit order fence Kansai+Hokuriku+Chubu avec thermal residual median
fence_cfg = next(cfg for cfg in configs if cfg.name == "Kansai+Hokuriku+Chubu | thermal_residual")
mo_fence = merit_order_for_areas(result, areas=list(fence_cfg.areas))
fence_setters = config_results[fence_cfg.name]
median_demand = fence_setters["demand_gw"].median()
median_spot = fence_setters["spot_eur_mwh"].median()

fig_fence = plot_merit_order_plotly(
    mo_fence,
    region=list(fence_cfg.areas),
    period=result.delivery_month,
    demand_gw=median_demand,
    spot_price_eur_mwh=median_spot,
    title=f"Merit Order ({', '.join(fence_cfg.areas)}) - thermal residual - {result.delivery_month:%B %Y}",
)
fig_fence.show()